# demo_team

> Phase 6 patch world: layouts, references, bandit vs random teams, circuit crowding.

Spec: `specs/multiagent/phase6/` (design-low-level section 9). Artifacts under `results/multiagent/demo_team/{quick,full}/` (gitignored): PNGs, `team.jsonl`, `metrics.json`, and **`report.md`** (summary). Quick: 300 s team run; `#| slow`: 600 s team run, 1200 s circuit, scaling preview.

In [ ]:
from explRL.core import loadCircuitAgent
from explRL.envs import ZoneArena
from explRL.members import CircuitMember
from explRL.multi import runTeam, spawnRngs
from explRL.patchworld import TeamPatchWorld, ringLayout, torusLayout
from explRL.team_agents import banditAgent, randomAgent, CircuitPatchMember
from explRL.team_refs import (idealFree, socialOptimum, ledgerArrays, welfare, gini,
    occupancy, coverage, dangerShare, ifdDistance, timeToIFD, welfareLoss)
from explRL.circuit_train import historyFromRecords, piScore
from explRL.train import demo_artifact_dir, save_demo_figure, save_demo_metrics, save_demo_report
import matplotlib.pyplot as plt
import numpy as np

DN = 'multiagent/demo_team'
artifact_mode, Ttot, circuit_T = 'quick', 300.0, 300.0
seed, nAgents, world_seed = 0, 8, 0
MAIN = 'ring'


In [ ]:
#| slow
artifact_mode, Ttot, circuit_T = 'full', 600.0, 1200.0


## Layouts

In [ ]:
def plot_layout(world, title, ax):
    lay = world.layout
    L = lay.T.shape[0]
    if lay.name == 'ring':
        th = np.linspace(0, 2 * np.pi, L, endpoint=False)
        xy = np.c_[np.cos(th), np.sin(th)]
    else:
        xy = lay.pos
    ax.scatter(xy[:, 0], xy[:, 1], c='0.85', s=30, zorder=1)
    for p, loc in enumerate(world.site):
        ax.scatter(xy[loc, 0], xy[loc, 1], s=80 + 40 * world.intake[p], c='C2', zorder=3)
    for loc in world.danger:
        ax.scatter(xy[loc, 0], xy[loc, 1], s=60, c='C3', marker='x', zorder=3)
    ax.set_title(title); ax.set_aspect('equal'); ax.axis('off')

ring_w = TeamPatchWorld(ringLayout(32), nPatches=4, patchSites=[2, 10, 18, 26], dangerSites=[6, 14, 22, 30], Tmove=None)
tor_w = TeamPatchWorld(torusLayout(8), nPatches=4, patchSites=[0, 7, 56, 63], dangerSites=[8, 15, 48, 55], Tmove=None)
ring_w.reset(0, 0); tor_w.reset(0, 0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_layout(ring_w, 'ring', axes[0]); plot_layout(tor_w, 'torus', axes[1])
plt.tight_layout()
save_demo_figure('layouts', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()


## References

In [ ]:
def ref_table(model):
    lay = ringLayout(32) if MAIN == 'ring' else torusLayout(8)
    kw = dict(layout=lay, nPatches=4, nDanger=4, model=model, Tmove=120.0,
              patchSites=[2, 10, 18, 26] if MAIN == 'ring' else [0, 7, 56, 63],
              dangerSites=[6, 14, 22, 30] if MAIN == 'ring' else [8, 15, 48, 55])
    w = TeamPatchWorld(**kw)
    out = {}
    for N in (4, 16, 64):
        n_ifd, y_ifd = idealFree(w, N)
        n_so, y_so = socialOptimum(w, N)
        out[str(N)] = dict(ifd_alloc=n_ifd.tolist(), ifd_yield=y_ifd, so_alloc=n_so.tolist(), so_yield=y_so)
    return out

references = {m: ref_table(m) for m in ('fixed', 'logistic')}
references


## Team run (bandit vs random)

In [ ]:
def default_world():
    lay = ringLayout(32) if MAIN == 'ring' else torusLayout(8)
    return TeamPatchWorld(lay, nPatches=4, nDanger=4, model='fixed', Tmove=120.0,
                          patchSites=[2, 10, 18, 26] if MAIN == 'ring' else [0, 7, 56, 63],
                          dangerSites=[6, 14, 22, 30] if MAIN == 'ring' else [8, 15, 48, 55])

def run_team(kind):
    w = default_world()
    w.reset(world_seed, nAgents)
    rngs = spawnRngs(seed, nAgents)
    members = [banditAgent() if kind == 'bandit' else randomAgent(r) for r in rngs]
    hist = runTeam(members, w, Ttot)
    return w, hist

def team_metrics(w, hist):
    L = ledgerArrays(w.ledger)
    tot, pas = welfare(L, nAgents, Ttot)
    harvest_by_agent = [float(L['harvest'][L['agent'] == i].sum()) for i in range(nAgents)]
    times, counts = occupancy(L, nAgents, w.nPatches, Ttot)          # 1 s grid (FR-19)
    n_ifd, _ = idealFree(w, nAgents)
    d = ifdDistance(counts, n_ifd, nAgents)
    moves = [k * w.Tmove for k in range(1, int(Ttot // w.Tmove) + 1) if k * w.Tmove < Ttot] if w.Tmove else []
    _, y_so = socialOptimum(w, nAgents)
    bins = np.arange(0.0, Ttot + 10.0, 10.0)
    rate = np.histogram(L['t'], bins=bins, weights=L['harvest'])[0] / 10.0   # team harvest/s, 10 s bins
    return dict(welfare=tot, perAgentSecond=pas, gini=gini(np.array(harvest_by_agent)),
                coverage=coverage(counts), dangerShare=dangerShare(L, w.danger, nAgents, Ttot),
                timeToIFD=timeToIFD(times, d, moves), welfareLoss=welfareLoss(tot, y_so, Ttot),
                times=times, ifd_d=d, moves=moves, bins=bins[:-1], rate=rate, so=y_so)

w_b, h_b = run_team('bandit')
w_r, h_r = run_team('random')
m_b, m_r = team_metrics(w_b, h_b), team_metrics(w_r, h_r)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for m, label in ((m_b, 'bandit'), (m_r, 'random')):
    axes[0].step(m['bins'], m['rate'], where='post', label=label)
    axes[1].plot(m['times'], m['ifd_d'], label=label)
axes[0].axhline(m_b['so'], color='k', ls=':', label='SO yield')
axes[0].set_ylabel('team harvest / s'); axes[0].legend(fontsize=8)
axes[1].set_ylabel('IFD distance'); axes[1].set_xlabel('time (s)')
for ax in axes:
    for mt in m_b['moves']:
        ax.axvline(mt, color='0.5', lw=0.8, ls='--')
plt.tight_layout()
save_demo_figure('team_run', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()

path = h_b.toJSONL(demo_artifact_dir(artifact_mode, name=DN) / 'team.jsonl')
print('wrote', path.name)


## Circuit team

In [ ]:
cw = TeamPatchWorld.zoneConfig(symmetric=False, intake=1.0)
cw.reset(seed, 4)
cmembers = [CircuitPatchMember(loadCircuitAgent(seed=seed + i)) for i in range(4)]
cteam = runTeam(cmembers, cw, circuit_T)
harvest = [float(sum(r.extras[-1] for r in cteam.forAgent(i))) for i in range(4)]   # extras[-1] = harvest
pis = [float(piScore(historyFromRecords(cteam.forAgent(i)))) for i in range(4)]
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(range(4), harvest); axes[0].set_ylabel('total harvest')
axes[1].bar(range(4), pis); axes[1].set_ylabel('PI')
for ax in axes:
    ax.set_xlabel('agent')
fig.suptitle('Circuit team with crowding (zoneConfig, intake 1, asymmetric)')
plt.tight_layout()
save_demo_figure('circuit_team', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()
circuit_team = dict(harvest=harvest, PI=pis)
circuit_team


## Sanity bridge (AC-1)

In [ ]:
CIRC_N = len(CircuitMember.extraNames)
sanity = {}
for symmetric in (True, False):
    ok = True
    for s in (0, 1, 2):
        env = ZoneArena(symmetric=symmetric)
        world = TeamPatchWorld.zoneConfig(symmetric=symmetric)
        world.reset(s, 1)
        h0 = runTeam([CircuitMember(loadCircuitAgent(seed=s))], env, 300.0)
        world.reset(s, 1)
        h1 = runTeam([CircuitPatchMember(loadCircuitAgent(seed=s))], world, 300.0)
        ok &= len(h0.records) == len(h1.records)
        for a, b in zip(h0.records, h1.records):
            ok &= (a.t, a.dt, a.location, a.reward) == (b.t, b.dt, b.location, b.reward)
            ok &= a.extras[:CIRC_N] == b.extras[:CIRC_N]
    sanity[str(symmetric)] = ok
print('sanity bridge:', sanity)


## Scaling preview (full run only)

In [ ]:
#| slow
SIZES = (1, 4, 16, 64)
NSEED = 10

def rates(kind, N, sd):
    "(welfare per agent-second over the run, team harvest/s in the first quarter)."
    w = default_world(); w.reset(sd, N)
    rngs = spawnRngs(seed + sd, N)
    members = [banditAgent() if kind == 'bandit' else randomAgent(r) for r in rngs]
    runTeam(members, w, Ttot)
    L = ledgerArrays(w.ledger)
    q = L['t'] < Ttot / 4
    return welfare(L, N, Ttot)[1], float(L['harvest'][q].sum()) / (Ttot / 4)

def boot_ci(vals, n=2000):
    rng = np.random.default_rng(0)
    vals = np.asarray(vals)
    boots = [rng.choice(vals, len(vals), replace=True).mean() for _ in range(n)]
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

scaling = {}
for N in SIZES:
    wref = default_world(); _, y_so = socialOptimum(wref, N); _, y_ifd = idealFree(wref, N)
    row = dict(so_yield=y_so, ifd_yield=y_ifd)
    for kind in ('bandit', 'random'):
        pas, firstQ = np.array([rates(kind, N, sd) for sd in range(NSEED)]).T
        row.update({f'{kind}_mean': float(pas.mean()), f'{kind}_ci': boot_ci(pas),
                    f'{kind}_firstQuarter': float(firstQ.mean())})
    scaling[str(N)] = row

# prototype checks re-run on the package world (AC-12): (1) no ceiling, (2) no floor
check1 = all(scaling[str(N)][f'{k}_firstQuarter'] < 0.9 * scaling[str(N)]['so_yield']
             for N in SIZES for k in ('bandit', 'random'))
check2 = all(scaling[str(N)]['bandit_ci'][0] > scaling[str(N)]['random_ci'][1] for N in SIZES)
scaling['check1_pass'] = check1
scaling['check2_pass'] = check2
print('check (1) no ceiling:', check1, ' check (2) no floor:', check2)

fig, ax = plt.subplots(figsize=(8, 4))
xs = np.array(SIZES)
for kind in ('bandit', 'random'):
    m = np.array([scaling[str(x)][f'{kind}_mean'] for x in xs])
    lo, hi = np.array([scaling[str(x)][f'{kind}_ci'] for x in xs]).T
    ax.errorbar(xs, m, yerr=[m - lo, hi - m], capsize=3, label=kind)
ax.plot(xs, [scaling[str(x)]['so_yield'] / x for x in xs], 'k--', label='SO yield / N')
ax.plot(xs, [scaling[str(x)]['ifd_yield'] / x for x in xs], 'k:', label='IFD yield / N')
ax.set_xscale('log', base=2); ax.set_xlabel('team size'); ax.set_ylabel('welfare / agent-s'); ax.legend()
save_demo_figure('scaling', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()
scaling


## Metrics export

In [ ]:
SERIES = ('times', 'ifd_d', 'moves', 'bins', 'rate', 'so')
team_run = {name: {k: v for k, v in m.items() if k not in SERIES} for name, m in (('bandit', m_b), ('random', m_r))}
metrics = dict(mode=artifact_mode, seed=seed, layout=MAIN, model='fixed', scheme='individual',
               scene_symmetric=False, Tmove=120.0, Ttot_s=Ttot, circuit_T_s=circuit_T, nAgents=nAgents,
               references=references, team_run=team_run, circuit_team=circuit_team, sanity=sanity)
if artifact_mode == 'full':
    metrics['scaling'] = scaling

def team_demo_report(m):
    b, r = m['team_run']['bandit'], m['team_run']['random']
    lines = [
        f"# Phase 6 team demo ({m['mode']} run)\n",
        f"Ring world with four moving resource patches, four danger sites, patches jump every {m['Tmove']} s. "
        f"Eight bandit learners vs eight random agents on the same world seed; {m['Ttot_s']} s team run. "
        f"Rewards are **individual** (harvest at patches, cost at danger).\n",
        "## Figures in this folder\n",
        "- `layouts.png` — ring and torus with patches (size = intake) and danger sites.\n",
        "- `team_run.png` — team harvest rate (10 s bins) and distance from the ideal free distribution over time; dashed lines = patch moves.\n",
        "- `circuit_team.png` — four circuit agents on a crowded ring (harvest and performance index).\n",
    ]
    if m['mode'] == 'full':
        lines.append("- `scaling.png` — welfare per agent vs team size (1, 4, 16, 64), bandit vs random.\n")
    lines += [
        "- `team.jsonl` — event log for the bandit team.\n",
        "- `metrics.json` — numbers behind this report.\n",
        "## Main takeaways\n",
        f"1. **Learning beats chance.** Bandit agents earned about **{b['perAgentSecond']:.3f}** harvest per agent per second; random agents about **{r['perAgentSecond']:.4f}**. Random wandering barely finds patches after they move.\n",
        f"2. **Fairness.** Bandit Gini {b['gini']:.2f} vs random {r['gini']:.2f} (lower is more even sharing of harvest).\n",
        f"3. **Room to improve.** Bandit welfare loss vs the social optimum: {b['welfareLoss']:.2f} (0 = perfect); random {r['welfareLoss']:.2f}.\n",
        f"4. **Patch coverage.** Bandits used about {100 * b['coverage']:.0f}% of patch-time with someone on a patch; random {100 * r['coverage']:.1f}%.\n",
    ]
    tifd = b.get('timeToIFD') or []
    ok = [t for t in tifd if t == t]
    if ok:
        lines.append(f"5. **After moves**, bandits typically got back near the ideal distribution within **{min(ok):.0f}–{max(ok):.0f} s** (when they did; see `team_run.png`).\n")
    lines.append(f"6. **Circuit bridge.** Sanity check vs `ZoneArena`: {m['sanity']} (should be true for both scenes).\n")
    h = m['circuit_team']['harvest']
    lines.append(f"7. **Circuit crowding.** Four circuit agents on the same ring split harvest unevenly (totals {', '.join(f'{x:.0f}' for x in h)}), as expected when they compete for the same patches.\n")
    if 'scaling' in m:
        sc = m['scaling']
        lines += [
            "## Scaling (full run only)\n",
            f"- Prototype-style **ceiling** check on the real world: {'passed' if sc.get('check1_pass') else 'failed'} (first-quarter harvest not too close to the optimum).\n",
            f"- **Floor** check: {'passed' if sc.get('check2_pass') else 'failed'} (bandit clearly above random at all team sizes).\n",
        ]
        for N in ('1', '4', '16', '64'):
            row = sc[N]
            lines.append(f"- Team size {N}: bandit {row['bandit_mean']:.3f} vs random {row['random_mean']:.4f} harvest per agent-s "
                         f"(SO reference {row['so_yield'] / int(N):.3f} per agent).\n")
        if sc.get('check1_pass') and int(sc['64']['bandit_firstQuarter']) > 0.85 * sc['64']['so_yield']:
            lines.append("- At **64 agents**, first-quarter team harvest is close to the theoretical ceiling; later phases may need harder worlds for very large teams.\n")
    lines.append("\nRe-run: `nbdev-test --n-workers 0 nbs/demo_team.ipynb` (quick) or add `--flags slow` (full). Spec: `specs/multiagent/phase6/RUN.md`.\n")
    return ''.join(lines)

save_demo_metrics(metrics, mode=artifact_mode, demo_name=DN)
rpath = save_demo_report(team_demo_report(metrics), mode=artifact_mode, demo_name=DN)
print('wrote', rpath.name)
print('bandit vs random perAgentSecond:', m_b['perAgentSecond'], m_r['perAgentSecond'])
